In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图风格与中文字体
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取存在数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_price_area_data(city_name):
    """
    获取指定城市的总价与面积数据
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        total_price,
        area_sqm
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND total_price IS NOT NULL 
      AND area_sqm IS NOT NULL
      AND area_sqm > 0
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_area_errorbar(city):
    """绘制带误差线的面积条形图"""
    df = fetch_price_area_data(city)
    
    if df.empty or len(df) < 20:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 数据量不足，无法生成有效的面积分析图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 动态价格分箱 ---
    quantiles = df['total_price'].quantile([0.2, 0.4, 0.6, 0.8]).tolist()
    bins = [-1] + [q + (i * 0.0001) for i, q in enumerate(quantiles)] + [float('inf')]
    
    labels = [
        f'低价位\n(<{int(quantiles[0])}万)', 
        f'中低价\n({int(quantiles[0])}-{int(quantiles[1])}万)', 
        f'中等价\n({int(quantiles[1])}-{int(quantiles[2])}万)', 
        f'中高价\n({int(quantiles[2])}-{int(quantiles[3])}万)', 
        f'高价位\n(>{int(quantiles[3])}万)'
    ]
    df['price_tier'] = pd.cut(df['total_price'], bins=bins, labels=labels)

    # --- 2. 统计平均值与标准差 ---
    stats = df.groupby('price_tier')['area_sqm'].agg(['mean', 'std']).reset_index()
    stats['std'] = stats['std'].fillna(0)

    # --- 3. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(12, 7))
    
    x_pos = np.arange(len(stats['price_tier']))
    means = stats['mean']
    stds = stats['std']
    bars = ax.bar(
        x_pos, 
        means, 
        yerr=stds, 
        capsize=8, 
        color='#48C9B0',       
        edgecolor='#1ABC9C',   
        linewidth=1.5,
        error_kw={'ecolor': '#34495E', 'elinewidth': 2, 'capthick': 2},
        alpha=0.85
    )

    # --- 4. 标注与修饰 ---
    for i, bar in enumerate(bars):
        yval = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2, 
            yval + 2, 
            f'{yval:.1f} ㎡', 
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#2C3E50'
        )

    plt.title(f'[{city}] 各价格区间平均房源面积及波动范围', fontsize=16, pad=20)
    plt.xlabel('自适应价格区间', fontsize=12, labelpad=10)
    plt.ylabel('建筑面积 (平方米)', fontsize=12)
    
    # 设置 X 轴刻度标签
    ax.set_xticks(x_pos)
    ax.set_xticklabels(stats['price_tier'], fontsize=11)
    
    # 辅助网格线与去边框
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    sns.despine(left=True, top=True, right=True)

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='#48C9B0', lw=6, label='平均面积'),
        Line2D([0], [0], marker='|', color='#34495E', lw=2, markersize=15, markeredgewidth=2, label='波动范围 (±1标准差)')
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=10, frameon=True, shadow=True)

    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_area_errorbar(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    # 首次渲染
    with plot_output:
        plot_area_errorbar(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()